# Cuantización de Qwen3.5-4B a Q4_K_M con llama.cpp
Notebook para **Google Colab** (siguiendo el apunte *Cuantización e Inferencia Local de LLMs*).

Resultado: `qwen3.5-4b_q4_k_m.gguf` (~2,7 GB) que después registramos en Ollama con `MODELFILE.txt`.

> En Colab: *Entorno de ejecución → Cambiar tipo → CPU alta RAM* alcanza (no hace falta GPU para cuantizar).

## 1. Clonar y compilar llama.cpp

In [ ]:
!git clone https://github.com/ggml-org/llama.cpp
%cd llama.cpp
!cmake -B build
!cmake --build build --config Release -j 2 --target llama-quantize
%cd ..
!pip install -q -r llama.cpp/requirements.txt

## 2. Descargar el modelo base (SafeTensors) desde Hugging Face

In [ ]:
from huggingface_hub import snapshot_download

path = snapshot_download(repo_id="Qwen/Qwen3.5-4B", local_dir="qwen")
print(path)
!ls -lh qwen

## 3. Convertir SafeTensors → GGUF (sin cuantizar, BF16)

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py ./qwen/ --outfile ./qwen/model-bf16.gguf --outtype bf16
!ls -lh qwen/*.gguf

## 4. Cuantizar a Q4_K_M
- **Q4**: ~4 bits por peso · **K**: K-quants · **M**: variante Medium (equilibrio tamaño/calidad).

In [ ]:
!./llama.cpp/build/bin/llama-quantize ./qwen/model-bf16.gguf ./qwen/qwen3.5-4b_q4_k_m.gguf Q4_K_M
!ls -lh qwen/*.gguf

## 5. Comparar tamaños (para el informe)

In [ ]:
import os
for f in ["qwen/model-bf16.gguf", "qwen/qwen3.5-4b_q4_k_m.gguf"]:
    print(f, round(os.path.getsize(f)/1024**3, 2), "GiB")

## 6. Descargar el .gguf
Guardarlo en Google Drive (más estable que la descarga directa para archivos grandes).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!cp qwen/qwen3.5-4b_q4_k_m.gguf /content/drive/MyDrive/